# 개념 2. 외삽의 한계와 보완

**목표:** 선형 외삽이 틀리는 세 가지 경우를 보고, 각각 어떻게 보완하는지 안다. 이 보완 방법들이 뒤의 칼만 필터·확장 칼만 필터로 이어진다.

| 경우 | 보완 |
|---|---|
| 1. 멀리 갈수록 가정이 깨진다 | 새 측정이 오면 다시 시작, 너무 멀면 멈춤 |
| 2. 측정에 잡음이 있다 | 두 점 대신 여러 점을 섞기 → **칼만 필터** |
| 3. 변화가 직선이 아니다 | 가정(모델)을 바꾸기 → **확장 칼만 필터** |
| 덤. 차수를 마구 올리면 | 오히려 크게 틀린다 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"   # 한글 글꼴 (macOS)
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
def linear_extrapolate(x, x1, y1, x2, y2):
    """두 점 (x1, y1), (x2, y2)를 지나는 직선의 x 에서의 값 (선형 외삽)."""
    slope = (y2 - y1) / (x2 - x1)
    return y2 + slope * (x - x2)

## 1. 멀리 갈수록 가정이 깨진다

**예시: 아이 키.** 5살 110cm, 10살 140cm. 선형 외삽하면?

In [ ]:
for age in [12, 20, 40]:
    print(f"{age}살: {linear_extrapolate(age, 5, 110, 10, 140):.0f} cm")

40살에 3m가 넘는다. *"매년 같은 만큼 큰다"*는 가정이 사춘기 뒤에 깨지기 때문이다.

외삽 결과는 **가정이 맞는 동안만** 믿을 수 있다. 그래서 실제로는 새 측정이 오면 그 값에서 다시 외삽하고, 너무 오래 측정이 없으면 외삽을 멈춘다.

## 2. 측정에 잡음이 있다

**예시: 아침 기온.** 실제로는 매시간 1.5°C씩 오르는데, 온도계가 ±1°C 흔들린다. 마지막 **두 점만** 쓰면 기울기가 잡음에 크게 흔들린다.

In [ ]:
rng = np.random.default_rng(0)
t = np.arange(6, 12)                                # 6시~11시
true = lambda h: 10 + 1.5 * (h - 6)                 # 실제 기온
temp = true(t) + rng.normal(0, 1.0, len(t))         # 잰 기온

two_point = lambda h: linear_extrapolate(h, t[-2], temp[-2], t[-1], temp[-1])
slope_fit, intercept = np.polyfit(t, temp, 1)       # 여러 점에 가장 잘 맞는 직선 (최소제곱)
fit = lambda h: slope_fit * h + intercept

h = np.linspace(6, 16, 100)
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(h, true(h), color="lightgray", lw=4, label="실제")
ax.plot(t, temp, "o", color="k", label="잰 값")
ax.plot(h[h >= 11], two_point(h[h >= 11]), "--", label="마지막 두 점으로")
ax.plot(h, fit(h), "--", label="여러 점에 맞춘 직선")
ax.set_xlabel("시각"); ax.set_ylabel("기온 (°C)"); ax.grid(alpha=.3); ax.legend(); plt.show()

print(f"15시 실제 {true(15):.1f}°C | 두 점 {two_point(15):.1f}°C | 여러 점 {fit(15):.1f}°C")

여러 점을 쓰면 잡음이 서로 상쇄돼서 기울기가 덜 흔들린다.

이 생각을 *"새 측정이 올 때마다 조금씩 고쳐 가는"* 방식으로 만든 것이 **칼만 필터(개념 3)**다.

## 3. 변화가 직선이 아니다

**예시: 브레이크를 밟은 자동차.** 0·1·2초에 0m, 18m, 32m 지점을 지났다. 1초마다 가는 거리가 18m → 14m로 줄고 있다.

선형 외삽은 같은 속도로 계속 간다고 본다. *"속도가 일정하게 줄어든다"*로 가정을 바꾸면 2차식이 된다.

$$y = y_0 + v\,t + \tfrac12 a\,t^2$$

In [ ]:
ts = np.array([0, 1, 2]); xs = np.array([0, 18, 32])
a2, a1, a0 = np.polyfit(ts, xs, 2)                  # 세 점을 지나는 2차식
quad = lambda s: a2 * s**2 + a1 * s + a0
line = lambda s: linear_extrapolate(s, ts[-2], xs[-2], ts[-1], xs[-1])

v0, a = a1, 2 * a2
t_stop = -v0 / a
print(f"처음 속도 {v0:.0f} m/s, 가속도 {a:.0f} m/s²")
print(f"멈추는 시각 {t_stop:.1f}초, 멈추는 위치 {quad(t_stop):.1f} m")

s = np.linspace(0, 6, 100)
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(ts, xs, "o", color="k", ms=8, label="잰 값")
ax.plot(s[s >= 2], line(s[s >= 2]), "--", label="선형 외삽: 같은 속도로")
ax.plot(s[s <= t_stop], quad(s[s <= t_stop]), "--", label="2차: 감속 반영")
ax.set_xlabel("초"); ax.set_ylabel("위치 (m)"); ax.grid(alpha=.3); ax.legend(); plt.show()

가정을 실제 움직임에 맞게 바꾸면 훨씬 잘 맞는다. 배라면 *"곧게 간다"* 대신 *"일정하게 돈다"*로 가정을 바꾸는 것이고, 이것이 **확장 칼만 필터(개념 4)**로 이어진다.

단, 2차식도 가정일 뿐이다. 멈춘 5초가 지난 뒤에도 그대로 쓰면 차가 **뒤로 간다**고 나온다.

## 덤. 차수를 마구 올리면

점이 많으면 높은 차수의 식으로 점을 전부 지나게 할 수 있다. 범위 안에서는 그럴듯하지만 범위 밖에서는 크게 튄다.

In [ ]:
x = np.arange(8.0)
y = 2 * x + rng.normal(0, 1.5, len(x))      # 실제로는 직선 + 잡음

xx = np.linspace(0, 11, 200)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, y, "o", color="k", label="잰 값")
for deg in [1, 2, 6]:
    ax.plot(xx, np.polyval(np.polyfit(x, y, deg), xx), label=f"{deg}차식")
ax.axvline(x[-1], color="gray", lw=1); ax.text(x[-1] + 0.2, -5, "← 잰 범위 | 외삽 →", color="gray")
ax.set_ylim(-10, 40); ax.grid(alpha=.3); ax.legend(); plt.show()

6차식은 잡음까지 흐름으로 믿어 버려서 범위를 벗어나자마자 엉뚱한 곳으로 간다. 가정은 **실제 움직임을 설명하는 것**으로 골라야지, 점에 억지로 맞추면 안 된다.

## 직접 해 보기

1. 기온 예시에서 잡음 `1.0`을 `3.0`으로 키우면 "두 점"과 "여러 점" 중 어느 쪽이 더 크게 틀리나?
2. 자동차 예시에서 세 번째 값 `32`를 `36`으로 바꾸면 멈추는 위치는 어떻게 되나?

In [ ]:
# 여기서 직접 바꿔 보세요


## 정리

| 한계 | 보완 | 다음 |
|---|---|---|
| 멀리 갈수록 틀린다 | 새 측정에서 다시 시작, 너무 멀면 멈춤 | |
| 잡음 | 여러 측정을 섞는다 | 개념 3 칼만 필터 |
| 직선이 아니다 | 가정을 실제 움직임에 맞게 바꾼다 | 개념 4 확장 칼만 필터 |